In [64]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

import pickle

In [65]:
df = pd.read_csv('application_train.csv')
bureau = pd.read_csv('bureau.csv')
prev = pd.read_csv('previous_application.csv')

In [66]:

threshold = len(df) * 0.5
df = df.dropna(thresh=threshold, axis=1)

df = df.fillna(df.median(numeric_only=True))

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna(df[col].mode()[0])

In [67]:
df = pd.get_dummies(df, drop_first=True)

In [68]:
bureau_agg = bureau.groupby('SK_ID_CURR').agg({
    'SK_ID_BUREAU': 'count',
    'AMT_CREDIT_SUM': 'sum',
    'AMT_CREDIT_SUM_DEBT': 'sum',
    'CREDIT_DAY_OVERDUE': 'sum'
}).rename(columns={
    'SK_ID_BUREAU': 'TOTAL_LOANS',
    'AMT_CREDIT_SUM': 'TOTAL_CREDIT',
    'AMT_CREDIT_SUM_DEBT': 'TOTAL_DEBT',
    'CREDIT_DAY_OVERDUE': 'TOTAL_OVERDUE'
}).reset_index()

In [69]:
df = df.merge(bureau_agg, on='SK_ID_CURR', how='left')

df[['TOTAL_LOANS','TOTAL_CREDIT','TOTAL_DEBT','TOTAL_OVERDUE']] = \
df[['TOTAL_LOANS','TOTAL_CREDIT','TOTAL_DEBT','TOTAL_OVERDUE']].fillna(0)

In [70]:
prev_agg = prev.groupby('SK_ID_CURR').agg({
    'SK_ID_PREV': 'count'
}).rename(columns={
    'SK_ID_PREV': 'TOTAL_APPLICATIONS'
}).reset_index()

df = df.merge(prev_agg, on='SK_ID_CURR', how='left')
df['TOTAL_APPLICATIONS'] = df['TOTAL_APPLICATIONS'].fillna(0)

In [71]:
df['DEBT_TO_CREDIT'] = df['TOTAL_DEBT'] / (df['TOTAL_CREDIT'] + 1)
df['CREDIT_PER_LOAN'] = df['TOTAL_CREDIT'] / (df['TOTAL_LOANS'] + 1)

df[['DEBT_TO_CREDIT','CREDIT_PER_LOAN']] = \
df[['DEBT_TO_CREDIT','CREDIT_PER_LOAN']].fillna(0)

In [72]:
X = df.drop('TARGET', axis=1)
y = df['TARGET']

In [73]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [74]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [75]:
model = LogisticRegression(max_iter=2000, class_weight='balanced')
model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=2000)

In [76]:
y_prob = model.predict_proba(X_test)[:,1]

# threshold tuning
y_pred = (y_prob > 0.6).astype(int)

In [77]:
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.95      0.82      0.88     25640
           1       0.20      0.50      0.28      2310

    accuracy                           0.79     27950
   macro avg       0.57      0.66      0.58     27950
weighted avg       0.89      0.79      0.83     27950

ROC-AUC: 0.7355953900493682


In [80]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Logistic (already done)
log_model = LogisticRegression(max_iter=2000, class_weight='balanced')
log_model.fit(X_train, y_train)
log_prob = log_model.predict_proba(X_test)[:,1]



In [81]:
#Random Forest
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
rf_prob = rf.predict_proba(X_test)[:,1]

In [82]:
# Gradient Boosting
gb = GradientBoostingClassifier()
gb.fit(X_train, y_train)
gb_prob = gb.predict_proba(X_test)[:,1]

In [83]:
from sklearn.metrics import roc_auc_score

print("Logistic ROC:", roc_auc_score(y_test, log_prob))
print("Random Forest ROC:", roc_auc_score(y_test, rf_prob))
print("Gradient Boost ROC:", roc_auc_score(y_test, gb_prob))

Logistic ROC: 0.7355953900493682
Random Forest ROC: 0.7132247452235752
Gradient Boost ROC: 0.742551841684057


In [84]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate(prob):
    pred = (prob > 0.6).astype(int)
    return (
        precision_score(y_test, pred),
        recall_score(y_test, pred),
        f1_score(y_test, pred)
    )

print("Logistic:", evaluate(log_prob))
print("RF:", evaluate(rf_prob))
print("GB:", evaluate(gb_prob))

Logistic: (0.19856581867850434, 0.5034632034632035, 0.2848047018489041)
RF: (0.0, 0.0, 0.0)
GB: (0.6470588235294118, 0.004761904761904762, 0.009454232917920068)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [85]:
print("Logistic:", evaluate(log_prob))
print("RF:", evaluate(rf_prob))
print("GB:", evaluate(gb_prob))

Logistic: (0.19856581867850434, 0.5034632034632035, 0.2848047018489041)
RF: (0.0, 0.0, 0.0)
GB: (0.6470588235294118, 0.004761904761904762, 0.009454232917920068)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [86]:
import pickle

pickle.dump(model, open('model.pkl', 'wb'))
pickle.dump(scaler, open('scaler.pkl', 'wb'))